# Week 4-5: Uncertainty Monitoring System Integration Test

This notebook tests the complete uncertainty monitoring pipeline built in Weeks 4-5:

1. **Measurement Strategies** - When to measure entropy
2. **UncertaintyMonitor** - Real-time uncertainty detection
3. **SpikeDetector** - Dynamic baseline spike detection

We'll integrate these with the Week 3 probe for end-to-end testing.

---

## Setup Instructions (Colab)

1. Upload this notebook to Colab
2. Upload `week3_probe.joblib` and `week3_scaler.joblib` when prompted
3. Run all cells

In [ ]:
# Cell 1: Install dependencies
!pip install -q transformers torch accelerate scipy scikit-learn pandas matplotlib seaborn joblib sentence-transformers

In [ ]:
# Cell 2: Clone repo and setup paths
import os

# Clone the repo
if not os.path.exists('/content/reposynth'):
    !git clone https://github.com/aniJani/reposynth.git /content/reposynth
    !cd /content/reposynth && git checkout Research
    print("Repo cloned successfully!")
else:
    !cd /content/reposynth && git pull
    print("Repo already exists, pulled latest changes.")

# Create Week3Results directory
!mkdir -p /content/reposynth/research/Week3Results

print("\nSetup complete!")

In [ ]:
# Cell 3: Upload Week 3 model files
from google.colab import files
import shutil

# Check if files already exist
probe_exists = os.path.exists('/content/reposynth/research/Week3Results/week3_probe.joblib')
scaler_exists = os.path.exists('/content/reposynth/research/Week3Results/week3_scaler.joblib')

if not probe_exists or not scaler_exists:
    print("Please upload the Week 3 model files:")
    print("  - week3_probe.joblib")
    print("  - week3_scaler.joblib")
    print()
    uploaded = files.upload()
    
    # Move files to correct location
    for filename in uploaded.keys():
        dest = f'/content/reposynth/research/Week3Results/{filename}'
        shutil.move(filename, dest)
        print(f"Moved {filename} to {dest}")
else:
    print("Week 3 model files already exist!")

# Verify files
print("\nVerifying files:")
!ls -la /content/reposynth/research/Week3Results/

In [ ]:
# Cell 4: Imports
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Tuple
import joblib

# Add project path for local imports
sys.path.insert(0, '/content/reposynth/packages/python-orchestrator')

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings('ignore')

print("Imports complete")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
# Cell 5: Import entropy module components
from orchestrator.entropy import (
    # Calculator
    EntropyCalculator,
    shannon_entropy,
    softmax,
    
    # Measurement strategies
    MeasurementContext,
    EveryTokenStrategy,
    SemanticBoundaryStrategy,
    LineStartStrategy,
    create_strategy,
    
    # Monitor
    UncertaintyMonitor,
    UncertaintyResult,
    create_monitor,
    
    # Spike detector
    SpikeDetector,
    MonitoredSpikeDetector,
    create_spike_detector,
)

print("Entropy module imports successful!")
print(f"Available strategies: EveryToken, SemanticBoundary, LineStart")

## 1. Test Measurement Strategies

In [ ]:
# Cell 6: Test measurement strategies

# Simulate generation
sample_generation = [
    "def", " ", "calculate", "_", "total", "(", "items", "):", "\n",
    "    ", "return", " ", "sum", "(", "items", ")", "\n"
]

strategies = {
    'every_token': EveryTokenStrategy(),
    'semantic_boundary': SemanticBoundaryStrategy(),
    'line_start': LineStartStrategy(),
}

print("Testing measurement strategies on sample generation:\n")
print(f"Tokens: {sample_generation}\n")

for name, strategy in strategies.items():
    generated = ""
    tokens_so_far = []
    measure_positions = []
    
    for i, token in enumerate(sample_generation):
        context = MeasurementContext(
            token=token,
            position=i,
            generated_so_far=generated,
            previous_tokens=tokens_so_far.copy(),
        )
        
        if strategy.should_measure(context):
            measure_positions.append(i)
        
        generated += token
        tokens_so_far.append(token)
    
    coverage = len(measure_positions) / len(sample_generation) * 100
    print(f"{name}:")
    print(f"  Positions: {measure_positions}")
    print(f"  Coverage: {coverage:.1f}% ({len(measure_positions)}/{len(sample_generation)} tokens)")
    print(f"  Expected: {strategy.get_expected_overhead()}")
    print()

## 2. Test Spike Detector

In [ ]:
# Cell 7: Test spike detector with synthetic data

np.random.seed(42)

# Generate synthetic entropy trace with spikes
n_samples = 50
baseline_entropy = 2.5  # Normal entropy level
noise_std = 0.3

# Base entropy with noise
entropy_trace = baseline_entropy + np.random.normal(0, noise_std, n_samples)

# Add 3 spikes
spike_positions = [10, 25, 40]
spike_values = [5.5, 6.0, 4.8]
for pos, val in zip(spike_positions, spike_values):
    entropy_trace[pos] = val

print("Synthetic entropy trace with 3 spikes at positions:", spike_positions)
print(f"Baseline: ~{baseline_entropy}, Spike values: {spike_values}")
print()

# Test different detection methods
methods = [
    ('threshold', {'threshold': 4.0}),
    ('relative', {'relative_factor': 1.5}),
    ('statistical', {'k_sigma': 2.0}),
    ('percentile', {'percentile': 95}),
    ('adaptive', {}),
]

print("Detection Results:")
print("-" * 60)

for method_name, params in methods:
    detector = SpikeDetector(method=method_name, **params)
    
    detected_spikes = []
    for i, value in enumerate(entropy_trace):
        detector.update(value, i)
        if detector.is_spike(value, i):
            detected_spikes.append(i)
    
    # Calculate precision and recall
    true_positives = len(set(detected_spikes) & set(spike_positions))
    precision = true_positives / len(detected_spikes) if detected_spikes else 0
    recall = true_positives / len(spike_positions)
    
    print(f"{method_name}:")
    print(f"  Detected: {detected_spikes}")
    print(f"  Precision: {precision:.1%}, Recall: {recall:.1%}")
    print()

In [ ]:
# Cell 8: Visualize spike detection

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for ax, (method_name, params) in zip(axes.flat, methods[:4]):
    detector = SpikeDetector(method=method_name, **params)
    
    detected = []
    baselines = []
    
    for i, value in enumerate(entropy_trace):
        detector.update(value, i)
        is_spike = detector.is_spike(value, i)
        if is_spike:
            detected.append(i)
        baselines.append(detector.get_baseline() or 0)
    
    # Plot
    ax.plot(entropy_trace, 'b-', alpha=0.7, label='Entropy')
    ax.plot(baselines, 'g--', alpha=0.5, label='Baseline')
    
    # Mark true spikes
    ax.scatter(spike_positions, [entropy_trace[p] for p in spike_positions],
               c='red', s=100, marker='o', label='True spikes', zorder=5)
    
    # Mark detected spikes
    if detected:
        ax.scatter(detected, [entropy_trace[p] for p in detected],
                   c='green', s=150, marker='x', linewidths=3,
                   label='Detected', zorder=6)
    
    ax.set_title(f'{method_name.capitalize()} Method')
    ax.set_xlabel('Position')
    ax.set_ylabel('Entropy')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('Week4_5_spike_detection.png', dpi=150)
plt.show()
print("Figure saved: Week4_5_spike_detection.png")

## 3. Load Model and Week 3 Probe

In [ ]:
# Cell 9: Load model
MODEL_NAME = "codellama/CodeLlama-7b-Instruct-hf"
print(f"Loading {MODEL_NAME}...")
print("This may take a few minutes...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
    output_hidden_states=True
)
model.eval()

print(f"Model loaded on {model.device}")

In [ ]:
# Cell 10: Load Week 3 probe

PROBE_PATH = '/content/reposynth/research/Week3Results/week3_probe.joblib'
SCALER_PATH = '/content/reposynth/research/Week3Results/week3_scaler.joblib'

probe_model = joblib.load(PROBE_PATH)
probe_scaler = joblib.load(SCALER_PATH)

# Week 3 best config
PROBE_LAYERS = [16, 24, 31]
PROBE_THRESHOLD = 0.06

print(f"Loaded Week 3 probe")
print(f"  Layers: {PROBE_LAYERS}")
print(f"  Threshold: {PROBE_THRESHOLD}")
print(f"  Probe type: {type(probe_model).__name__}")

In [ ]:
# Cell 11: Helper functions for hidden state extraction

def get_hidden_states(text: str, layers: List[int]) -> np.ndarray:
    """Extract and concatenate hidden states from specified layers."""
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
    
    states = [
        outputs.hidden_states[layer_idx + 1][:, -1, :].cpu().numpy()[0]
        for layer_idx in layers
    ]
    return np.concatenate(states).astype(np.float32)

def get_logits(text: str) -> np.ndarray:
    """Get logits for next token prediction."""
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.logits[0, -1, :].cpu().numpy()

print("Helper functions ready")

## 4. Test UncertaintyMonitor with Real Model

In [ ]:
# Cell 12: Test uncertainty monitor with different methods

# Test prompts - one that should trigger retrieval (code context needed)
# and one that shouldn't (natural language)
test_prompts = [
    {
        'prompt': 'In our React app, authentication is done using',
        'expected': 'CODE',
        'should_retrieve': True,
    },
    {
        'prompt': 'The weather today is',
        'expected': 'LANGUAGE',
        'should_retrieve': False,
    },
]

print("Testing UncertaintyMonitor with different methods:\n")

methods_to_test = ['raw_entropy', 'normalized_entropy', 'prob_differential']

for prompt_info in test_prompts:
    prompt = prompt_info['prompt']
    print(f"\nPrompt: '{prompt}'")
    print(f"Expected: {prompt_info['expected']} (retrieve={prompt_info['should_retrieve']})")
    print("-" * 60)
    
    # Get logits for the prompt
    logits = get_logits(prompt)
    
    for method in methods_to_test:
        # Create monitor with appropriate threshold
        if method == 'raw_entropy':
            threshold = 5.0  # High entropy = uncertain
        elif method == 'normalized_entropy':
            threshold = 0.5
        else:
            threshold = 0.9
        
        monitor = create_monitor(
            method=method,
            threshold=threshold,
            tokenizer=tokenizer,
            strategy='every_token',
        )
        
        # Simulate single step
        result = monitor.step(logits, prompt.split()[-1])
        
        if result:
            status = "RETRIEVE" if result.should_retrieve else "continue"
            print(f"  {method}: value={result.value:.4f}, threshold={threshold}, -> {status}")

In [ ]:
# Cell 13: Test with probe-based monitor

from orchestrator.entropy import create_probe_monitor

print("Testing probe-based monitor (Week 3 best method):\n")

for prompt_info in test_prompts:
    prompt = prompt_info['prompt']
    print(f"Prompt: '{prompt}'")
    print(f"Expected: {prompt_info['expected']}")
    
    # Get hidden states
    hidden_states = get_hidden_states(prompt, PROBE_LAYERS)
    logits = get_logits(prompt)
    
    # Create probe monitor
    monitor = create_probe_monitor(
        probe_model=probe_model,
        probe_scaler=probe_scaler,
        probe_layers=PROBE_LAYERS,
        threshold=PROBE_THRESHOLD,
    )
    
    # Run step
    result = monitor.step(logits, prompt.split()[-1], hidden_states=hidden_states)
    
    if result:
        status = "RETRIEVE" if result.should_retrieve else "continue"
        print(f"  Probe score: {result.value:.4f}, threshold={PROBE_THRESHOLD}, -> {status}")
        correct = (status == "RETRIEVE") == prompt_info['should_retrieve']
        print(f"  Correct: {correct}")
    print()

## 5. End-to-End Generation with Monitoring

In [ ]:
# Cell 14: Simulate generation with uncertainty monitoring

def monitored_generation(
    prompt: str,
    max_tokens: int = 20,
    monitor_method: str = 'raw_entropy',
    threshold: float = 5.0,
):
    """
    Generate tokens while monitoring uncertainty.
    Returns the generation and entropy trace.
    """
    # Create monitor
    monitor = create_monitor(
        method=monitor_method,
        threshold=threshold,
        tokenizer=tokenizer,
        strategy='semantic_boundary',
        max_retrievals=3,
    )
    
    generated_text = prompt
    generated_tokens = []
    entropy_trace = []
    
    for i in range(max_tokens):
        # Get logits for next token
        inputs = tokenizer(generated_text, return_tensors="pt").to(model.device)
        with torch.no_grad():
            outputs = model(**inputs)
        logits = outputs.logits[0, -1, :].cpu().numpy()
        
        # Sample next token
        probs = softmax(logits)
        next_token_id = np.argmax(probs)
        next_token = tokenizer.decode([next_token_id])
        
        # Monitor step
        result = monitor.step(logits, next_token)
        
        if result:
            entropy_trace.append({
                'position': i,
                'value': result.value,
                'token': next_token,
                'should_retrieve': result.should_retrieve,
            })
        
        # Update generation
        generated_text += next_token
        generated_tokens.append(next_token)
        
        # Stop on EOS
        if next_token in ['<s>', '</s>', '<|endoftext|>']:
            break
    
    return {
        'prompt': prompt,
        'generation': generated_text[len(prompt):],
        'tokens': generated_tokens,
        'entropy_trace': entropy_trace,
        'summary': monitor.get_summary(),
    }

print("Monitored generation function ready")

In [ ]:
# Cell 15: Run monitored generation on test cases

test_cases = [
    "For the REST API, the framework we use is",
    "The weather today is",
    "In our React app, authentication is done using",
    "My favorite color has always been",
]

results = []
for prompt in test_cases:
    print(f"\nGenerating for: '{prompt}'")
    result = monitored_generation(prompt, max_tokens=15)
    results.append(result)
    
    print(f"  Generated: {result['generation'][:50]}...")
    print(f"  Measurements: {result['summary']['measurements_taken']}")
    print(f"  Retrievals triggered: {result['summary']['retrievals_triggered']}")
    
    if result['entropy_trace']:
        print(f"  Mean uncertainty: {result['summary'].get('mean_uncertainty', 0):.3f}")
        print(f"  Max uncertainty: {result['summary'].get('max_uncertainty', 0):.3f}")

In [ ]:
# Cell 16: Visualize entropy traces

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, result in zip(axes.flat, results):
    trace = result['entropy_trace']
    if not trace:
        ax.text(0.5, 0.5, 'No measurements', ha='center', va='center')
        ax.set_title(result['prompt'][:40] + '...')
        continue
    
    positions = [t['position'] for t in trace]
    values = [t['value'] for t in trace]
    retrievals = [t['position'] for t in trace if t['should_retrieve']]
    
    ax.plot(positions, values, 'b-o', alpha=0.7, markersize=5)
    
    if retrievals:
        retrieval_values = [trace[positions.index(p)]['value'] for p in retrievals]
        ax.scatter(retrievals, retrieval_values, c='red', s=100, marker='v',
                   label='Retrieval triggered', zorder=5)
    
    ax.axhline(y=5.0, color='r', linestyle='--', alpha=0.5, label='Threshold')
    ax.set_xlabel('Token Position')
    ax.set_ylabel('Entropy')
    ax.set_title(result['prompt'][:40] + '...')
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper right')

plt.tight_layout()
plt.savefig('Week4_5_entropy_traces.png', dpi=150)
plt.show()
print("Figure saved: Week4_5_entropy_traces.png")

## 6. Test MonitoredSpikeDetector (Combined Pipeline)

In [ ]:
# Cell 17: Test combined monitor + spike detector

from orchestrator.entropy import create_monitored_detector

detector = create_monitored_detector(
    uncertainty_method='raw_entropy',
    spike_method='statistical',
    tokenizer=tokenizer,
    measurement_strategy='every_token',
    k_sigma=2.0,
    max_retrievals=3,
)

# Run on a test prompt
prompt = "For the REST API, the framework we use is"
generated_text = prompt

print(f"Prompt: '{prompt}'")
print("\nGeneration with spike detection:")
print("-" * 60)

for i in range(20):
    inputs = tokenizer(generated_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits[0, -1, :].cpu().numpy()
    
    probs = softmax(logits)
    next_token_id = np.argmax(probs)
    next_token = tokenizer.decode([next_token_id])
    
    result = detector.step(logits, next_token)
    
    if result:
        is_spike = result.details.get('is_spike', False)
        spike_str = "SPIKE!" if is_spike else ""
        print(f"  [{i:2d}] '{next_token}' entropy={result.value:.3f} {spike_str}")
        
        if is_spike and 'spike_info' in result.details:
            info = result.details['spike_info']
            print(f"       -> severity={info['severity']}, baseline={info['baseline']:.3f}")
    
    generated_text += next_token
    
    if next_token in ['</s>', '<|endoftext|>']:
        break

print("\nSummary:")
summary = detector.get_summary()
print(f"  Total tokens: {summary['total_tokens']}")
print(f"  Retrievals triggered: {summary['retrievals_triggered']}")
print(f"  Detector spike count: {summary['detector']['spike_count']}")

## 7. Summary & Next Steps

In [ ]:
# Cell 18: Summary

print("="*70)
print("WEEK 4-5 IMPLEMENTATION SUMMARY")
print("="*70)

print("""
Components Implemented:

1. MEASUREMENT STRATEGIES (measurement.py)
   - EveryTokenStrategy: 100% coverage (baseline)
   - SemanticBoundaryStrategy: ~10-20% coverage (our approach)
   - LineStartStrategy: ~5-15% coverage (UnCert-CoT style)
   - NthTokenStrategy: Configurable periodic sampling
   - CompositeStrategy: Combine multiple strategies

2. UNCERTAINTY MONITOR (monitor.py)
   - Supports 5 methods: raw_entropy, normalized_entropy,
     prob_differential, cce, probe
   - Real-time step-by-step monitoring
   - History tracking and visualization
   - Configurable max retrievals

3. SPIKE DETECTOR (spike_detector.py)
   - 5 detection methods: threshold, relative, statistical,
     percentile, adaptive
   - Dynamic baseline calculation
   - Confidence scoring and severity levels
   - Debouncing to prevent rapid retrieval

4. COMBINED PIPELINE (MonitoredSpikeDetector)
   - Integrates monitor + detector
   - Unified interface for generation loop

Next Steps (Phase 3: Adaptive Context Retrieval):
- Implement TopicInferrer (what to retrieve)
- Implement AdaptiveContextRetriever
- Implement ContextManager (budget management)
- Full integration with RepoSynth retrieval
""")

print("="*70)

In [ ]:
# Cell 19: Save results

import json

week4_5_results = {
    'components_implemented': [
        'measurement.py - Measurement strategies',
        'monitor.py - UncertaintyMonitor',
        'spike_detector.py - SpikeDetector',
    ],
    'measurement_strategies': ['every_token', 'semantic_boundary', 'line_start', 'every_n', 'composite'],
    'uncertainty_methods': ['raw_entropy', 'normalized_entropy', 'prob_differential', 'cce', 'probe'],
    'spike_methods': ['threshold', 'relative', 'statistical', 'percentile', 'adaptive'],
    'integration_test': 'passed',
    'week3_probe_integration': 'working',
}

with open('Week4_5_results.json', 'w') as f:
    json.dump(week4_5_results, f, indent=2)

print("Results saved to Week4_5_results.json")

# Download results if in Colab
try:
    from google.colab import files
    files.download('Week4_5_results.json')
    files.download('Week4_5_spike_detection.png')
    files.download('Week4_5_entropy_traces.png')
except:
    pass